# Toy manifolds — tangent geometry from a diffusion score

**Question.** A diffusion score $v(x,\sigma) = x + \sigma^2 s(x,\sigma)$
(*Landing with the Score*, Kharitenko, Shen, De Santi, He, Dörfler,
arXiv:2509.23357) is trained with **no geometric supervision** -- it only
ever sees noisy points and learns to denoise them. Does it nevertheless
learn the **intrinsic dimension** and **local geometry** of the data
manifold, well enough to attempt building an explicit low-dimensional
chart from it?

**What this notebook shows, in order:**
1. A torus toy dataset with known ground truth, and a trained score.
2. A quick sanity check: does the score actually denoise onto the manifold?
3. Unsupervised dimension detection ($k=2$, never told to the model).
4. Local tangent coherence (holonomy) -- a short stepping stone.
5. **The main attempt**: building a global 2D chart via a tangential
   Delaunay triangulation + LTSA (Local Tangent Space Alignment). We show
   this **fails** in two specific, informative ways (no periodicity
   closure, no injectivity) -- understanding *why* it fails is the main
   contribution here, not a polished final chart.
6. A simpler contrasting example -- a **circle** ($S^1\subset\mathbb R^2$,
   intrinsic dimension $k=1$) -- where a much simpler chart-building
   approach (direct integration of the tangent field) works well,
   because the key difficulty found on the torus (rotational degeneracy
   of a 2D tangent eigenspace) is structurally absent in 1D.
7. Open questions.


## 1. Setup

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from scipy.spatial import Delaunay
from collections import Counter
from mpl_toolkits.mplot3d.art3d import Line3DCollection
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from torch.utils.data import DataLoader, Dataset
from scipy.stats import vonmises
from scipy.special import logsumexp

import data
from data import TorusDataset
from models.scoreMLP import ScoreMLP, score, v, v_jvp, full_jacobian
import train as T
from tangent_geometry import (detect_k, tangent_basis, filtered_neighbors,
                                delaunay_neighbors, procrustes_rotation,
                                synchronize_orientation, holonomy_error,
                                triangle_shape_quality, grassmann_distance,
                                build_alignment_matrix, global_coordinates)
from geodesic_chart import integrate_direction

%matplotlib inline
torch.manual_seed(0)


### Utility functions

Grouped here, reused across several sections below.

In [ ]:
def sample_uniform(n, seed):
    """Uniform draw on [0,2pi)^2 -- used for topology/structural geometry
    tests, where the mixture's density would create sampling gaps
    indistinguishable from genuine topological holes."""
    rng = np.random.default_rng(seed)
    return rng.uniform(0, 2*np.pi, size=(n, 2))


def torus_3d(angles, R=2.0, r=1.0):
    """Standard 3D torus embedding (donut), for visualization only --
    this is NOT the actual ambient embedding (R^4) used everywhere else
    for computation."""
    th1, th2 = angles[:, 0], angles[:, 1]
    x = (R + r*np.cos(th2)) * np.cos(th1)
    y = (R + r*np.cos(th2)) * np.sin(th1)
    z = r * np.sin(th2)
    return np.column_stack([x, y, z])


def edge_consistency(N, epsilon=0.25, n_presel=150, seed=0):
    """% of edges shared by exactly 2 triangles, on the exact ground truth."""
    angles_n = data.sample(N, seed=seed)
    x_n_np = data.embed(angles_n)
    x_n = torch.tensor(x_n_np, dtype=torch.float32)
    P_n = torch.tensor(data.tangent_projector(x_n_np), dtype=torch.float32)
    k_n = detect_k(P_n, energy_threshold=0.9)
    B_n = tangent_basis(P_n, k_n)

    raw_neighbors = filtered_neighbors(x_n, B_n, epsilon=epsilon, n_presel=n_presel)

    seen_triangles = set()
    for i in range(N):
        idx = raw_neighbors[i]
        patch_idx = torch.cat([torch.tensor([i]), idx])
        diffs = x_n[patch_idx] - x_n[i]
        local_coords = (diffs @ B_n[i]).detach().numpy()
        try:
            tri = Delaunay(local_coords)
        except Exception:
            continue
        for simplex in tri.simplices:
            if 0 in simplex:
                g = [patch_idx[v].item() for v in simplex]
                seen_triangles.add(tuple(sorted(g)))

    edge_count = Counter()
    for (a, b, c) in seen_triangles:
        for e in [(a, b), (b, c), (a, c)]:
            edge_count[tuple(sorted(e))] += 1
    counts = Counter(edge_count.values())
    n_clean = counts.get(2, 0)
    n_total = sum(counts.values())
    return 100 * n_clean / n_total


def holonomy_summary(x, jacobians, k, epsilon=0.25, n_presel=100, label=""):
    """Holonomy (deviation from identity) on Delaunay triangles, after
    orientation synchronization."""
    B = tangent_basis(jacobians, k)
    raw_neighbors = filtered_neighbors(x, B, epsilon=epsilon, n_presel=n_presel)
    B_sync = synchronize_orientation(B, raw_neighbors)
    d_neighbors = delaunay_neighbors(x, B_sync, raw_neighbors)

    seen = set()
    errors = []
    N = x.shape[0]
    for i in range(N):
        idx = d_neighbors[i]
        for j_pos in range(len(idx)):
            for l_pos in range(j_pos+1, len(idx)):
                j, l = idx[j_pos].item(), idx[l_pos].item()
                key = tuple(sorted([i, j, l]))
                if key in seen:
                    continue
                if j not in d_neighbors[l].tolist() and l not in d_neighbors[j].tolist():
                    continue
                seen.add(key)
                Rij = procrustes_rotation(B_sync[i], B_sync[j])
                Rjl = procrustes_rotation(B_sync[j], B_sync[l])
                Rli = procrustes_rotation(B_sync[l], B_sync[i])
                errors.append(holonomy_error(Rij, Rjl, Rli))

    errors = np.array(errors)
    print(f"{label}: {len(errors)} triangles, median holonomy={np.median(errors):.2e}, "
          f"mean={errors.mean():.2e}, max={errors.max():.2e} rad")
    return errors


## 2. Torus toy dataset and score training

### 2.1 Decor -- the exact density

In [ ]:
grid_n = 200
th = np.linspace(0, 2*np.pi, grid_n, endpoint=False)
TH1, TH2 = np.meshgrid(th, th, indexing='ij')
angles_grid = np.column_stack([TH1.ravel(), TH2.ravel()])
U = data.potential(angles_grid, data.KAPPA).reshape(grid_n, grid_n)

fig, ax = plt.subplots(figsize=(5, 5))
c = ax.contourf(th, th, U.T, levels=40, cmap='viridis')
plt.colorbar(c, ax=ax, label=r'$U(\theta_1,\theta_2) = -\log p$')
ax.set_xlabel(r'$\theta_1$')
ax.set_ylabel(r'$\theta_2$')
ax.set_title('Exact potential on the torus')
plt.tight_layout()
plt.show()


### 2.2 Training (ambient noise)

Warning: `n_epochs` below is a quick demo value. For the real results
referenced throughout this notebook (k=2, near-zero holonomy), use a much
larger budget (tens of thousands of steps).

In [ ]:
ds = TorusDataset(n=5000, seed=0)
loader = DataLoader(ds, batch_size=T.BATCH_SIZE, shuffle=True)
model_ambient, losses_ambient = T.train(loader, dim=4, n_epochs=60, seed=0, log_every=50)

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(losses_ambient)
ax.set_yscale('log')
ax.set_xlabel('step'); ax.set_ylabel('CSM loss')
ax.set_title('Training loss (ambient noise)')
plt.tight_layout()
plt.show()


## 3. Sanity check: does the score actually denoise onto the manifold?

Before looking at anything geometric, the most basic question: given a
point corrupted by ambient Gaussian noise, does $v(x,\sigma,\text{model})$
actually move it back towards the torus?

This uses `data.project`, an EXACT, independent geometric function (the
true nearest point on the torus for any ambient point) -- not derived
from the noisy or denoised points themselves, so the comparison is not
circular.

In [ ]:
sigma_test = 0.3
N_sanity = 300

angles_clean = data.sample(N_sanity, seed=5)
x_clean_np = data.embed(angles_clean)
x_clean = torch.tensor(x_clean_np, dtype=torch.float32)

noise = torch.randn_like(x_clean) * sigma_test
x_noisy = x_clean + noise

sigma_t = torch.full((N_sanity,), sigma_test)
x_denoised = v(x_noisy, sigma_t, model_ambient)

dist_before = np.linalg.norm(x_noisy.detach().numpy() - data.project(x_noisy.detach().numpy()), axis=1)
dist_after = np.linalg.norm(x_denoised.detach().numpy() - data.project(x_denoised.detach().numpy()), axis=1)

print(f"mean distance to the torus BEFORE denoising: {dist_before.mean():.4f}")
print(f"mean distance to the torus AFTER  denoising: {dist_after.mean():.4f}")

# Visual check, in REAL ambient coordinates (never forced via unembed):
# plot 3 of the 4 true ambient coordinates, for the exact torus surface
# (from a grid of KNOWN angles) and for the noisy/denoised points.
grid_small = np.linspace(0, 2*np.pi, 40, endpoint=False)
GA, GB = np.meshgrid(grid_small, grid_small, indexing='ij')
surface_angles = np.column_stack([GA.ravel(), GB.ravel()])
surface_x = data.embed(surface_angles)

fig = plt.figure(figsize=(12, 6))
for i, (pts, title) in enumerate([(x_noisy.detach().numpy(), 'Before (noisy)'),
                                    (x_denoised.detach().numpy(), 'After (denoised)')]):
    ax = fig.add_subplot(1, 2, i+1, projection='3d')
    ax.scatter(surface_x[:,0], surface_x[:,1], surface_x[:,2], s=1, alpha=0.05, color='gray')
    ax.scatter(pts[:,0], pts[:,1], pts[:,2], s=6, color='crimson', alpha=0.7)
    ax.set_title(title)
    ax.set_xlabel('coord 0'); ax.set_ylabel('coord 1'); ax.set_zlabel('coord 2')
plt.tight_layout()
plt.show()


## 4. Unsupervised dimension detection

The model was only ever shown noisy points in $\mathbb R^4$ -- it was
never told the data lies on a 2-dimensional surface.

In [ ]:
sigma_val = 0.06
N_diag = 500
angles_diag = data.sample(N_diag, seed=1)
x_diag_np = data.embed(angles_diag)
x_diag = torch.tensor(x_diag_np, dtype=torch.float32)
sigma_diag = torch.full((N_diag,), sigma_val)

jacobians_learned = full_jacobian(x_diag, sigma_diag, model_ambient)
k_learned = detect_k(jacobians_learned, energy_threshold=0.9)
print(f"k detected on the trained score: {k_learned}")

P_exact = torch.tensor(data.tangent_projector(x_diag_np), dtype=torch.float32)
k_exact = detect_k(P_exact, energy_threshold=0.9)
print(f"k detected on the exact ground truth: {k_exact}")


## 5. Holonomy -- local tangent coherence

A stepping stone before attempting a global chart: is the tangent basis
extracted at neighboring points *consistent* with itself (parallel
transport around a small triangle returns close to the identity), rather
than just individually plausible?

In [ ]:
errors_exact = holonomy_summary(x_diag, P_exact, k_exact, label="Exact ground truth")
errors_learned = holonomy_summary(x_diag, jacobians_learned, k_learned, label="Trained score")


## 6. Main attempt: building a global 2D chart

Given the tangent field is locally coherent (section 5), the natural next
step is to try to build an explicit map $\Phi^{-1}:\mathcal M\to\mathbb
R^2$. We build a tangential Delaunay triangulation (from the learned/exact
tangent bases) and run **LTSA** (Local Tangent Space Alignment, Zhang &
Zha 2004) on it.

**This section documents a failure, on purpose.** Understanding precisely
*how* and *why* it fails is the most informative part of this notebook.

### 6.1 Robustness of the local triangulation to anchor density

Before even attempting LTSA, a prerequisite: the triangulation itself
must be combinatorially consistent (every edge shared by exactly 2
triangles). This is checked on the exact ground truth, and improves with
anchor density -- though never reaches 100% with this simplified
construction (see discussion below).

Warning: expensive cell (tens of seconds) -- reduce `Ns` for a quick test.

In [ ]:
Ns = [3000, 6000, 12000]
percentages = [edge_consistency(N) for N in Ns]

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(Ns, percentages, 'o-')
ax.set_xlabel('N (number of anchors)')
ax.set_ylabel('% combinatorially clean edges')
ax.set_title('Local triangulation consistency vs. density')
ax.set_xscale('log')
plt.tight_layout()
plt.show()

for N, p in zip(Ns, percentages):
    print(f"N={N}: {p:.2f}% clean edges")


### 6.2 3D visualization of the triangulation

Uniform sampling (not `data.sample`): used to visually check the
resulting triangulation, independent of the mixture's sampling gaps.

In [ ]:
N_viz = 4000
angles_viz = sample_uniform(N_viz, seed=2)
x_viz_np = data.embed(angles_viz)
x_viz = torch.tensor(x_viz_np, dtype=torch.float32)
P_viz = torch.tensor(data.tangent_projector(x_viz_np), dtype=torch.float32)
k_viz = detect_k(P_viz, energy_threshold=0.9)
B_viz = tangent_basis(P_viz, k_viz)

raw_neighbors_viz = filtered_neighbors(x_viz, B_viz, epsilon=0.25, n_presel=100)
d_neighbors_viz = delaunay_neighbors(x_viz, B_viz, raw_neighbors_viz)
xyz = torus_3d(angles_viz)

edges = set()
for i in range(N_viz):
    for j in d_neighbors_viz[i].tolist():
        edges.add(tuple(sorted([i, j])))
segments = [[xyz[a], xyz[b]] for a, b in edges]

fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(projection='3d')
ax.scatter(xyz[:,0], xyz[:,1], xyz[:,2], s=2, color='black', alpha=0.4)
lc = Line3DCollection(segments, colors='steelblue', linewidths=0.3, alpha=0.4)
ax.add_collection3d(lc)
ax.set_title('Tangential Delaunay triangulation -- uniform sampling, N=4000')
ax.set_box_aspect([1,1,0.5])
plt.tight_layout()
plt.show()


### 6.3 LTSA global coordinates, and where it breaks

Two DISTINCT failure modes, found empirically:
1. **No periodicity closure**: the two ends of a non-contractible loop
   ($\theta_1\approx 0$ and $\theta_1\approx 2\pi$, physically identical
   on the torus) get assigned different LTSA coordinates -- the map
   "cuts" the torus into a cylinder rather than closing it.
2. **No injectivity guarantee** (distinct from 1): pairs of points that
   are physically FAR apart on the torus can end up close together in the
   LTSA coordinates, purely as an artifact of the least-squares
   optimization -- with no theoretical guarantee preventing it.

In [ ]:
raw_neighbors_ltsa = filtered_neighbors(x_viz, B_viz, epsilon=0.25, n_presel=100)
d_neighbors_ltsa = delaunay_neighbors(x_viz, B_viz, raw_neighbors_ltsa)

S = build_alignment_matrix(x_viz, B_viz, d_neighbors_ltsa)
Y = global_coordinates(S, k=2)

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
axes[0].scatter(angles_viz[:, 0], angles_viz[:, 1], c=angles_viz[:, 0], cmap='hsv', s=5)
axes[0].set_xlabel(r'$\theta_1$'); axes[0].set_ylabel(r'$\theta_2$')
axes[0].set_title('True angles (color = theta_1)')
axes[1].scatter(Y[:, 0], Y[:, 1], c=angles_viz[:, 0], cmap='hsv', s=5)
axes[1].set_xlabel('LTSA coord 1'); axes[1].set_ylabel('LTSA coord 2')
axes[1].set_title('LTSA coordinates (same color = same true theta_1)')
plt.tight_layout()
plt.show()


In [ ]:
# Quantify failure mode 2 (injectivity): pairs close in Y but far in true theta.
from scipy.spatial.distance import pdist, squareform

D_Y = squareform(pdist(Y))

def angular_dist(a1, a2):
    d1 = np.minimum(np.abs(a1[:,0:1]-a2[:,0].T), 2*np.pi-np.abs(a1[:,0:1]-a2[:,0].T))
    d2 = np.minimum(np.abs(a1[:,1:2]-a2[:,1].T), 2*np.pi-np.abs(a1[:,1:2]-a2[:,1].T))
    return np.sqrt(d1**2 + d2**2)

D_theta = angular_dist(angles_viz, angles_viz)

N_pts = len(angles_viz)
iu = np.triu_indices(N_pts, k=1)
close_in_Y = D_Y[iu] < np.percentile(D_Y[iu], 1)
far_in_theta = D_theta[iu] > np.percentile(D_theta[iu], 50)
n_violations = (close_in_Y & far_in_theta).sum()
print(f"pairs close in Y but far in true theta: {n_violations} / {close_in_Y.sum()} "
      f"(of the 1%-closest pairs in Y)")


## 7. A simpler contrast: the circle ($S^1$, $k=1$)

Why does the torus resist a clean chart? One concrete, isolable reason:
the 2D tangent eigenspace has a genuine **rotational degeneracy** (both
eigenvalues equal), so the eigenbasis returned at each point is only
defined up to an arbitrary in-plane rotation -- this rotation must be
tracked and aligned consistently, which is exactly where LTSA (a global
least-squares method) has no guarantee, and where even direct tangent-field
integration (`geodesic_chart.py`) needs an explicit local Procrustes
realignment step to stay consistent (see that module's docstring for a
concrete bug found and fixed during development).

In 1D, there is no such rotational freedom: the tangent line only has a
sign ambiguity, trivially handled. This section repeats the same
tangent-field-integration test on a circle, to isolate the effect.

In [ ]:
CIRCLE_MU = np.array([0.0, np.pi])
CIRCLE_WEIGHTS = np.array([0.6, 0.4])
CIRCLE_KAPPA = 1.0    # moderate contrast: density ratio ~1.8 (max/min)

def circle_log_density(theta):
    z = np.log(CIRCLE_WEIGHTS) + vonmises.logpdf(theta[:, None], CIRCLE_KAPPA, loc=CIRCLE_MU)
    return logsumexp(z, axis=1)

def sample_circle(n, seed):
    rng = np.random.default_rng(seed)
    component = rng.choice(2, size=n, p=CIRCLE_WEIGHTS)
    theta = vonmises.rvs(CIRCLE_KAPPA, loc=CIRCLE_MU[component], random_state=rng)
    return theta % (2 * np.pi)

def embed_circle(theta):
    return np.column_stack([np.cos(theta), np.sin(theta)])

def unembed_circle(x_np):
    return np.arctan2(x_np[:, 1], x_np[:, 0]) % (2 * np.pi)

def tangent_projector_circle(x_np):
    n = x_np / np.linalg.norm(x_np, axis=1, keepdims=True)
    N = x_np.shape[0]
    I = np.eye(2)[None, :, :].repeat(N, axis=0)
    outer = n[:, :, None] * n[:, None, :]
    return I - outer

def project_circle(x_np):
    return x_np / np.linalg.norm(x_np, axis=1, keepdims=True)

class CircleDataset(Dataset):
    def __init__(self, n, seed):
        theta = sample_circle(n, seed)
        self.points = torch.tensor(embed_circle(theta), dtype=torch.float32)
    def __len__(self):
        return self.points.shape[0]
    def __getitem__(self, i):
        return self.points[i]


### 7.1 Dimension detection and training

In [ ]:
theta_test = sample_circle(500, seed=0)
x_circle_np = embed_circle(theta_test)
P_circle = torch.tensor(tangent_projector_circle(x_circle_np), dtype=torch.float32)
k_circle = detect_k(P_circle, energy_threshold=0.9)
print(f"k detected on the circle (exact ground truth): {k_circle}")

ds_circle = CircleDataset(n=5000, seed=0)
loader_circle = DataLoader(ds_circle, batch_size=T.BATCH_SIZE, shuffle=True)
model_circle, losses_circle = T.train(loader_circle, dim=2, n_epochs=2000, seed=0, log_every=500)

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(losses_circle)
ax.set_yscale('log')
ax.set_xlabel('step'); ax.set_ylabel('CSM loss')
ax.set_title('Training loss (circle, ambient noise)')
plt.tight_layout()
plt.show()


### 7.2 Loop-closure test via tangent-field integration

Instead of a discrete triangulation, integrate the (1D) tangent direction
as an ODE, all the way around the circle, and measure how far the
endpoint lands from the start -- this should be near-zero if the learned
tangent field is coherent all the way around.

In [ ]:
sigma_val_circle = 0.06

def jacobian_fn_circle_learned(x):
    sigma_t = torch.full((x.shape[0],), sigma_val_circle)
    return full_jacobian(x, sigma_t, model_circle)

def retract_fn_circle_learned(x):
    sigma_t = torch.full((x.shape[0],), sigma_val_circle)
    return v(x, sigma_t, model_circle)

def jacobian_fn_circle_exact(x):
    x_np = x.detach().numpy()
    return torch.tensor(tangent_projector_circle(x_np), dtype=torch.float32)

def retract_fn_circle_exact(x):
    x_np = x.detach().numpy()
    return torch.tensor(project_circle(x_np), dtype=torch.float32)

step_size = 0.05
n_steps = int(2 * np.pi / step_size)

closures_exact, closures_learned = [], []
for seed in range(10):
    theta0 = sample_circle(1, seed=seed + 100)
    x0 = torch.tensor(embed_circle(theta0), dtype=torch.float32)

    path_exact = integrate_direction(x0, 0, jacobian_fn_circle_exact, retract_fn_circle_exact,
                                       k=1, step_size=step_size, n_steps=n_steps, retract_every=1)
    path_learned = integrate_direction(x0, 0, jacobian_fn_circle_learned, retract_fn_circle_learned,
                                         k=1, step_size=step_size, n_steps=n_steps, retract_every=1)

    closures_exact.append(np.linalg.norm(path_exact[-1].detach().numpy() - path_exact[0].detach().numpy()))
    closures_learned.append(np.linalg.norm(path_learned[-1].detach().numpy() - path_learned[0].detach().numpy()))

print("Loop closure, EXACT ground truth: min/median/max:",
      f"{min(closures_exact):.4f} / {sorted(closures_exact)[len(closures_exact)//2]:.4f} / {max(closures_exact):.4f}")
print("Loop closure, LEARNED score:      min/median/max:",
      f"{min(closures_learned):.4f} / {sorted(closures_learned)[len(closures_learned)//2]:.4f} / {max(closures_learned):.4f}")


**Comparison note (see project discussion for the full torus numbers,
not re-run here for time):** on the torus ($k=2$), the same
tangent-field-integration test gave a *median* loop closure around 1.2
(a sizeable fraction of the torus's own scale) with a trained score of
comparable quality -- versus a much smaller and more consistent closure
here on the circle. This is consistent with the torus's difficulty coming
specifically from the $k\ge 2$ rotational degeneracy discussed above, not
merely from the score being insufficiently trained.

## 8. Open questions

- The torus chart-building attempt (section 6) fails in two specific,
  now well-characterized ways (periodicity closure, injectivity) -- both
  documented failure modes of LTSA-style methods in general, not specific
  to our tangent estimates.
- Two literature directions were identified as principled next steps but
  not implemented here: (a) discrete Ricci flow (Jin, Kim, Luo, Gu 2008)
  to obtain a provably-flat metric before unfolding, and (b) resolving
  star-inconsistency between neighboring tangential-Delaunay patches via
  weighted Delaunay / Moser-Tardos perturbation (Boissonnat & Ghosh 2014).
  Both were explored on paper but ran into either non-convergence
  (Ricci flow, on a small residual fraction of low-degree vertices) or
  substantial remaining implementation scope, given the time available.
- The circle example isolates the likely central obstacle (tangent-basis
  rotational degeneracy for $k\ge 2$) from the separate question of score
  quality -- suggesting that a chart-building method built around
  continuous tangent-field integration (rather than a discrete
  triangulation) may be the more promising direction for $k=2$, PROVIDED
  the rotational alignment along a path is handled carefully (as done
  here via local Procrustes realignment).
